# Задача 3. Цвета клеток картинками из MNIST

Среда `MnistGridWorldEnv` (`envs/mnist_env.py`) — копия обычной, только вместо one-hot цвета агент видит
**случайную картинку цифры**: цвет 2 -> случайная двойка из MNIST. Картинки уменьшены до 14x14 и вытянуты в вектор из 196 чисел.

Цифр всего 10, поэтому цветов пола не больше 7 (+3 служебных). Сетап 1 (25 уникальных цветов) здесь невозможен — берём сетапы 2, 3, 4.

In [ ]:
import csv
import os
from pathlib import Path

# работаем из корня репозитория: туда пишутся runs/, outputs/, data/
if Path.cwd().name == "src_v2":
    os.chdir(Path.cwd().parent)

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display

from gridrl_v2 import utils
from gridrl_v2.envs import load_mnist, make_env
from gridrl_v2.models import DSAC, PPO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("учимся на", DEVICE)

SETUPS = {
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

## Как выглядят наблюдения

В каждой строке — цвет клетки и 8 случайных картинок, которыми его может увидеть агент.
Для сетапа 3 (7 цветов пола): цифры 0–6 — пол, 7 — цель, 8 — препятствие, 9 — стена.

In [ ]:
images = load_mnist()  # при первом запуске скачает MNIST в data/mnist (~10 МБ)

fig, axes = plt.subplots(10, 8, figsize=(8, 10))
for digit in range(10):
    for j in range(8):
        picture = images[digit][np.random.randint(len(images[digit]))]
        axes[digit, j].imshow(picture.reshape(14, 14), cmap="gray")
        axes[digit, j].axis("off")
    axes[digit, 0].set_title(f"цвет {digit}", fontsize=8, loc="left")
fig.tight_layout()
Path("outputs/envs").mkdir(parents=True, exist_ok=True)
fig.savefig("outputs/envs/mnist_colors.png", dpi=120, bbox_inches="tight")
plt.show()

## PPO на MNIST-среде

Конфиг тот же, что в `test_task.ipynb`, плюс `"mnist": True`. Сеть не меняем: на вход теперь идёт вектор из 196 чисел,
и первый линейный слой сети сам учится «читать» цифры.

In [ ]:
ppo_cfg = {
    "device": DEVICE,
    "mnist": True,  # наблюдения картинками цифр
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "episodes": 1000,
    "episodes_per_update": 10,
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "run_name": "ppo_mnist",
}
PPO_EPISODES = {2: 1000, 3: 1500, 4: 1500}

ppo_algs = {}
for i in SETUPS:
    print(f"===== PPO, MNIST, Env{i} =====")
    alg = PPO({**ppo_cfg, **SETUPS[i], "episodes": PPO_EPISODES[i], "run_name": f"ppo_mnist_env{i}"})
    alg.train()
    alg.save()
    ppo_algs[f"Env{i}"] = alg

## DSAC на MNIST-среде

**Внимание:** DSAC учится долго, на картах 10x10 — долго, может быть час и больше на одну карту на CPU.

In [ ]:
dsac_cfg = {
    "device": DEVICE,
    "mnist": True,
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "soft_q": True,
    "episodes": 500,
    "buffer_episodes": 5000,
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,
    "lr": 3e-4,
    "alpha": 0.005,
    "hidden": 128,
    "run_name": "dsac_mnist",
}
DSAC_EPISODES = {2: 500, 3: 800, 4: 800}

dsac_algs = {}
for i in SETUPS:
    print(f"===== DSAC, MNIST, Env{i} =====")
    alg = DSAC({**dsac_cfg, **SETUPS[i], "episodes": DSAC_EPISODES[i], "run_name": f"dsac_mnist_env{i}"})
    alg.train()
    alg.save()
    dsac_algs[f"Env{i}"] = alg

## Графики: one-hot против MNIST

Прогоны с one-hot наблюдениями взяты из `test_task.ipynb` (`runs/ppo_env{i}`, `runs/dsac_env{i}`).
Линия — среднее за последние 50 эпизодов, полоса — ± стандартное отклонение.

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")


def load_history(run_name):
    # читаем metrics.csv: словарь {колонка: список чисел}; None, если прогона ещё нет
    path = Path("runs") / run_name / "metrics.csv"
    if not path.exists():
        print("нет прогона", run_name)
        return None
    with open(path) as f:
        rows = list(csv.DictReader(f))
    history = {}
    for key in rows[0]:
        values = []
        for row in rows:
            if row[key] == "True":
                values.append(1.0)
            elif row[key] == "False":
                values.append(0.0)
            else:
                values.append(float(row[key]))
        history[key] = values
    return history


def plot_curve(ax, values, label, color, window=50):
    # линия — среднее за последние window эпизодов, полоса — ± стандартное отклонение там же
    means = []
    stds = []
    for i in range(len(values)):
        last = [v for v in values[max(0, i - window + 1):i + 1] if not np.isnan(v)]
        if len(last) > 0:
            means.append(np.mean(last))
            stds.append(np.std(last))
        else:
            means.append(np.nan)
            stds.append(np.nan)
    means = np.array(means)
    stds = np.array(stds)
    episodes = np.arange(len(values))
    ax.plot(episodes, means, color=color, linewidth=2, label=label)
    ax.fill_between(episodes, means - stds, means + stds, color=color, alpha=0.3)

In [ ]:
for algo in ["ppo", "dsac"]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    for ax, i in zip(axes, SETUPS):
        onehot = load_history(f"{algo}_env{i}")
        mnist = load_history(f"{algo}_mnist_env{i}")
        if onehot is not None:
            plot_curve(ax, onehot["return"], "one-hot цвет", "tab:blue")
        if mnist is not None:
            plot_curve(ax, mnist["return"], "картинка MNIST", "tab:red")
        ax.set_title(f"{algo.upper()}, Env{i}")
        ax.set_xlabel("эпизод")
        ax.set_ylabel("средняя отдача")
        ax.legend()
    Path("outputs/plots").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"outputs/plots/mnist_{algo}_vs_onehot.png", dpi=150, bbox_inches="tight")
    plt.show()

## Как ходит обученный агент (PPO)

In [ ]:
for env_name, alg in ppo_algs.items():
    episodes = alg.evaluate(100)
    success = np.mean([ep["success"] for ep in episodes])
    length = np.mean([ep["length"] for ep in episodes])
    title = f"PPO MNIST {env_name}: success {success:.2f}, длина {length:.1f} (оптимум {alg.optimal['length']:.1f})"

    fig, ax = plt.subplots(figsize=(6, 6))
    image = utils.plot_heatmap(ax, alg.env, episodes)
    fig.colorbar(image, ax=ax, fraction=0.046, label="посещений за эпизод")
    ax.set_title(title, fontsize=9)
    Path("outputs/eval").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"outputs/eval/ppo_mnist_{env_name}_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()

    short = [ep for ep in episodes if ep["length"] <= 60][:5]
    if len(short) == 0:
        short = [episodes[0]]
    utils.save_gif(alg.env, short, f"outputs/eval/ppo_mnist_{env_name}.gif", f"PPO MNIST {env_name}")
    display(Image(filename=f"outputs/eval/ppo_mnist_{env_name}.gif"))

## Ответы на вопросы из задания

**На что влияет такое изменение среды и что оно тестирует?**
Раньше один цвет всегда давал одно и то же наблюдение. Теперь одному цвету соответствуют тысячи разных картинок,
и на каждом шаге приходит новая. Агенту приходится сначала *понять, какая это цифра* (по сути решить задачу
классификации внутри RL), и только потом ориентироваться. Это проверяет, умеет ли метод работать с более сложными
и зашумлёнными наблюдениями: распознавать одно и то же состояние в разных «обличьях» и не путаться от шума.
Обучение становится медленнее, потому что сигнал (награда) тот же редкий, а распознавать нужно больше.

**Нужна ли другая архитектура сети? Точно ли нужен серьёзный энкодер?**
Не нужен. Картинки маленькие (14x14), а различать надо всего 10 цифр — с этим справляется даже один линейный слой
(линейный классификатор на MNIST даёт около 92% точности). Поэтому сеть оставили прежней: первый слой
`Linear(196 + 4 -> 128)` + `Tanh` сам учится распознавать цифры. Свёрточная сеть дала бы чуть лучшее распознавание,
но усложнила бы код и замедлила обучение. *(Проверить по графикам выше: насколько MNIST медленнее one-hot.)*